# Normal Equation vs Ridge Regularization — Hands-on Experiment

**Goal**: Understand how Ridge regularization ($\alpha I$) shrinks weights by running the Normal Equation with different $\alpha$ values on a toy dataset.

**True relationship**: $y = x_1 + x_2$ (perfect linear, no noise) — so the ideal weights are $\theta = [1.0, 1.0]$.

In [1]:
import numpy as np

## 1. Setup — Toy Dataset

$$X = \begin{bmatrix} 1 & 2 \\ 2 & 3 \\ 3 & 4 \\ 4 & 5 \\ 5 & 6 \end{bmatrix}, \quad y = \begin{bmatrix} 3 \\ 5 \\ 7 \\ 9 \\ 11 \end{bmatrix}$$

5 samples, 2 features. $y = x_1 + x_2$ exactly.

In [2]:
X = np.array([[1,2], [2,3], [3,4], [4,5], [5,6]])
y = np.array([3, 5, 7, 9, 11])

print(f"X shape: {X.shape}")  # (5, 2) — 5 samples, 2 features
print(f"y shape: {y.shape}")  # (5,)   — 5 target values

X shape: (5, 2)
y shape: (5,)


## 2. Normal Equation (no regularization)

$$\theta = (X^T X)^{-1} X^T y$$

Step by step: compute $X^T$, then $X^T X$, then invert, then multiply by $X^T y$.

In [ ]:
XT = X.T        # calculating transpose of the X (.T is means transpose)
XTX = XT @ X
XTX_inv = np.linalg.inv(XTX)

theta_normal = XTX_inv @ XT @ y

print(f"X^T X =\n{XTX}\n")
print(f"(X^T X)^-1 =\n{XTX_inv}\n")
print(f"theta = {theta_normal}")
print(f"Expected: [1.0, 1.0] — perfect fit since y = x1 + x2")

X^T X =
[[55 70]
 [70 90]]

(X^T X)^-1 =
[[ 1.8 -1.4]
 [-1.4  1.1]]

theta = [1. 1.]
Expected: [1.0, 1.0] — perfect fit since y = x1 + x2


## 3. Ridge Regularization — Adding $\alpha I$

$$\theta_{ridge} = (X^T X + \alpha I)^{-1} X^T y$$

Adding $\alpha I$ increases the diagonal of $X^T X$ before inverting. Bigger matrix $\rightarrow$ smaller inverse $\rightarrow$ smaller weights.

**Why?** For a $2 \times 2$ matrix, the inverse scales by $\frac{1}{ad - bc}$. Making $a$ and $d$ bigger increases the denominator, shrinking the result.

Let's test with $\alpha = 1, 10, 100$:

In [4]:
def ridge_normal_equation(X, y, alpha):
    """Compute theta using Ridge Normal Equation: (X^T X + αI)^-1 X^T y"""
    n = X.shape[1]
    XTX = X.T @ X
    XTX_reg = XTX + alpha * np.identity(n)
    XTX_reg_inv = np.linalg.inv(XTX_reg)
    theta = XTX_reg_inv @ X.T @ y
    return theta, XTX_reg, XTX_reg_inv

print(f"{'Alpha':<8} {'X^T X + αI':<30} {'Inverse max val':<18} {'θ':<22} {'Fit'}")
print("-" * 100)

for alpha in [0, 1, 10, 100]:
    theta, XTX_reg, XTX_reg_inv = ridge_normal_equation(X, y, alpha)
    inv_max = np.max(np.abs(XTX_reg_inv))
    print(f"{alpha:<8} diag=[{XTX_reg[0,0]:.0f}, {XTX_reg[1,1]:.0f}]"
          f"{'':<16} {inv_max:<18.4f} [{theta[0]:.4f}, {theta[1]:.4f}]"
          f"{'':>4} {'Perfect' if alpha == 0 else 'Underfit' if alpha >= 10 else 'Close'}")

Alpha    X^T X + αI                     Inverse max val    θ                      Fit
----------------------------------------------------------------------------------------------------
0        diag=[55, 90]                 1.8000             [1.0000, 1.0000]     Perfect
1        diag=[56, 91]                 0.4643             [0.8929, 1.0714]     Close
10       diag=[65, 100]                 0.0625             [0.8125, 1.0313]     Underfit
100      diag=[155, 190]                 0.0077             [0.5112, 0.6538]     Underfit


## 4. Key Takeaway — Bias-Variance Tradeoff

| $\alpha$ | Effect | When it helps |
|----------|--------|---------------|
| 0 | No penalty — exact fit | Data is clean, true relationship is linear |
| Small | Slight shrinkage — more stable | Some noise, moderate number of features |
| Large | Heavy shrinkage — underfits | Only useful when weights are truly out of control (e.g. many polynomial features) |

**On this clean data** ($y = x_1 + x_2$, no noise), any $\alpha > 0$ hurts — the true weights are already small and correct. Ridge doesn't know that; it shrinks regardless.

**In the house price project**, Ridge with $\alpha = 1000$ helped because polynomial features (degree 2 on 15 features → 136 features) created many large, noisy weights that needed taming.